# 01 · Data collection
**Project:** Diwali, stubble burning and Delhi's air (DA2402 Data Curation, Project #2)

**Question:** How much of the October–November PM2.5 rise in Delhi comes from Diwali night, and how much from the season's weather?

This notebook collects all raw data and saves it, untouched, in `raw/`. It covers three sources:

| # | Source | What we take | How |
|---|---|---|---|
| 1 | **OpenAQ** (API v3 + AWS S3 archive) | Station list for Delhi; PM2.5 readings for 2025 | Scripted (this notebook) |
| 2 | **Open-Meteo Historical Weather API** | Hourly weather incl. boundary-layer height, 2023–2025 | Scripted (this notebook) |
| 3 | **CPCB CAAQMS Data Repository** | Hourly PM2.5 + station weather, 12 stations, 2023–2025 | Downloaded by hand (captcha); steps and file list recorded below, files checked by code |

Everything is cached: if a raw file already exists it is not downloaded again, so the notebook can be re-run top to bottom safely.

In [1]:
# --- Setup ---------------------------------------------------------------
import os, json, re, time
from datetime import date, datetime, timezone, timedelta
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import requests
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

# Project root = folder that contains raw/, interim/, clean/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, INTERIM, CLEAN, META = ROOT / "raw", ROOT / "interim", ROOT / "clean", ROOT / "metadata"
for p in [RAW / "openaq", RAW / "openmeteo", RAW / "cpcb", INTERIM, CLEAN, META]:
    p.mkdir(parents=True, exist_ok=True)

# OpenAQ API key: read from the environment or a local .env file (never written into the notebook).
# In Colab, store it under Secrets as OPENAQ_API_KEY.
def load_key():
    if os.environ.get("OPENAQ_API_KEY"):
        return os.environ["OPENAQ_API_KEY"]
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get("OPENAQ_API_KEY")
    except Exception:
        pass
    env = ROOT / ".env"
    if env.exists():
        for line in env.read_text().splitlines():
            if line.startswith("OPENAQ_API_KEY="):
                return line.split("=", 1)[1].strip()
    raise RuntimeError("Set OPENAQ_API_KEY (environment, Colab secret or .env file)")

OPENAQ_KEY = load_key()
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "DA2402-student-project (IIT Madras; course project)"
print("Project root:", ROOT)

Project root: /sessions/rcw-01a8zy71zkermwkg9fl4twy5/mnt/Delhi_pollution_level


## 1 · Study design
The brief fixes the city (Delhi) and three Diwalis. For each year we take a window from **15 September to 30 November**: the second half of September is the "before the season" baseline, and October–November is the season the question asks about.

In [2]:
DIWALI = {2023: date(2023, 11, 12), 2024: date(2024, 10, 31), 2025: date(2025, 10, 20)}
WINDOW = {y: (date(y, 9, 15), date(y, 11, 30)) for y in DIWALI}

# The 12 Delhi stations we use. CPCB name = exactly as written in the CPCB repository dropdown;
# CPCB site id = the number in the repository's download link (site_<id>_1Hr.csv).
# OpenAQ id = the location we believe is the same station in OpenAQ (matching is checked in notebook 02).
STATIONS = pd.DataFrame([
    ("anand_vihar",    "Anand Vihar, Delhi - DPCC",         301,  235,   "East"),
    ("rk_puram",       "R K Puram, Delhi - DPCC",           124,  17,    "South"),
    ("okhla_ph2",      "Okhla Phase-2, Delhi - DPCC",       1428, 8239,  "South"),
    ("punjabi_bagh",   "Punjabi Bagh, Delhi - DPCC",        125,  50,    "West"),
    ("najafgarh",      "Najafgarh, Delhi - DPCC",           1427, 10488, "West"),
    ("ito",            "ITO, Delhi - CPCB",                 117,  5613,  "Central"),
    ("pusa_dpcc",      "Pusa, Delhi - DPCC",                1563, 6356,  "Central"),
    ("pusa_iitm",      "Pusa, Delhi - IITM",                107,  5404,  "Central"),
    ("lodhi_road_imd", "IMD Lodhi Road, Delhi - IITM",      109,  5634,  "Central"),
    ("jahangirpuri",   "Jahangirpuri, Delhi - DPCC",        1423, 8235,  "North"),
    ("narela",         "Narela, Delhi - DPCC",              1426, 10485, "North"),
    ("dwarka_sec8",    "Dwarka-Sector 8, Delhi - DPCC",     1422, 6931,  "South-west"),
], columns=["station_key", "cpcb_name", "cpcb_site_id", "openaq_id", "zone"])
# One extra OpenAQ location that is a candidate duplicate of Lodhi Road (checked in notebook 02)
EXTRA_OPENAQ_IDS = [11607]
STATIONS

,station_key,cpcb_name,cpcb_site_id,openaq_id,zone
0,anand_vihar,"Anand Vihar, Delhi - DPCC",301,235,East
1,rk_puram,"R K Puram, Delhi - DPCC",124,17,South
2,okhla_ph2,"Okhla Phase-2, Delhi - DPCC",1428,8239,South
3,punjabi_bagh,"Punjabi Bagh, Delhi - DPCC",125,50,West
4,najafgarh,"Najafgarh, Delhi - DPCC",1427,10488,West
5,ito,"ITO, Delhi - CPCB",117,5613,Central
6,pusa_dpcc,"Pusa, Delhi - DPCC",1563,6356,Central
7,pusa_iitm,"Pusa, Delhi - IITM",107,5404,Central
8,lodhi_road_imd,"IMD Lodhi Road, Delhi - IITM",109,5634,Central
9,jahangirpuri,"Jahangirpuri, Delhi - DPCC",1423,8235,North


## 2 · robots.txt checks
Before collecting anything we read the robots.txt of every host we contact (status 404 = the site publishes none).

In [3]:
def now_utc():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

robots = []
for host in ["https://api.openaq.org", "https://openaq-data-archive.s3.amazonaws.com",
             "https://archive-api.open-meteo.com", "https://open-meteo.com",
             "https://airquality.cpcb.gov.in"]:
    r = SESSION.get(host + "/robots.txt", timeout=30)
    body = r.text.strip() if (r.status_code == 200 and "<html" not in r.text.lower()[:200]) else ""
    robots.append({"host": host, "status": r.status_code,
                   "robots.txt": body[:120] if body else "(no robots.txt published)",
                   "checked_utc": now_utc()})
pd.DataFrame(robots)

,host,status,robots.txt,checked_utc
0,https://api.openaq.org,404,(no robots.txt published),2026-11-03T09:44:32Z
1,https://openaq-data-archive.s3.amazonaws.com,404,(no robots.txt published),2026-11-03T09:44:33Z
2,https://archive-api.open-meteo.com,200,User-agent: *\nDisallow: /,2026-11-03T09:44:34Z
3,https://open-meteo.com,200,User-agent: *\nAllow: /,2026-11-03T09:44:35Z
4,https://airquality.cpcb.gov.in,404,(no robots.txt published),2026-11-03T09:44:35Z


**What this means**
- **OpenAQ** (API and S3 archive) publishes no robots.txt. Its terms of use require an API key and say users must use "provided APIs, export functions, or other officially sanctioned methods", and they forbid scraping. We use only the documented API and the public S3 archive, which OpenAQ itself documents.
- **archive-api.open-meteo.com** has `Disallow: /`. That blocks *crawlers* from indexing the API host; it does not forbid documented API calls. Open-Meteo's terms allow free non-commercial use (educational use is listed) under CC BY 4.0, with limits of 10,000 calls a day and 600 a minute. We make 3 calls.
- **airquality.cpcb.gov.in** publishes no robots.txt, but the data repository sits behind a captcha. We therefore **did not script it**: files were downloaded by hand in a browser (Section 5).

## 3 · OpenAQ
### 3a · Station list (API v3)
We ask the API for every monitoring location inside a box around Delhi NCT and save the raw JSON response.

In [4]:
OPENAQ_API = "https://api.openaq.org/v3"
BBOX = "76.84,28.40,77.35,28.89"   # lon_min, lat_min, lon_max, lat_max around Delhi NCT

loc_file = RAW / "openaq" / "openaq_locations_delhi_bbox.json"
if not loc_file.exists():
    r = SESSION.get(f"{OPENAQ_API}/locations", params={"bbox": BBOX, "limit": 1000},
                    headers={"X-API-Key": OPENAQ_KEY}, timeout=60)
    r.raise_for_status()
    loc_file.write_text(r.text, encoding="utf-8")

locs_json = json.loads(loc_file.read_text(encoding="utf-8"))
locs = pd.json_normalize(locs_json["results"])
print("Locations returned:", len(locs))
locs["has_pm25"] = locs["sensors"].apply(lambda s: any(x["parameter"]["name"] == "pm25" for x in s))
locs[["id", "name", "provider.name", "owner.name", "coordinates.latitude", "coordinates.longitude",
      "datetimeFirst.utc", "datetimeLast.utc", "has_pm25"]].sort_values("name").head(15)

Locations returned: 99


,id,name,provider.name,owner.name,coordinates.latitude,coordinates.longitude,datetimeFirst.utc,datetimeLast.utc,has_pm25
86,4712609,Air Check,AirGradient,FamilyFirst,28.623056,77.234167,2026-01-28T11:00:00Z,2026-10-31T20:00:00Z,True
51,6932,"Alipur, Delhi - DPCC",CPCB,Delhi Pollution Control Committee,28.815329,77.153010,2018-12-15T15:00:00Z,2026-10-31T18:00:00Z,True
88,6145551,Anand Lok,AirGradient,Unknown Person,28.558700,77.218860,2025-11-28T11:00:00Z,2026-10-31T20:00:00Z,True
73,10487,"Anand Vihar, Delhi - DPCC",caaqm,Unknown Governmental Organization,28.646835,77.316032,2018-08-13T11:00:00Z,2021-09-20T07:45:00Z,True
14,5509,"Anand Vihar, Delhi - DPCC",caaqm,Unknown Governmental Organization,28.647622,77.315809,2018-03-09T03:30:00Z,2022-10-31T00:30:00Z,True
6,235,"Anand Vihar, New Delhi - DPCC",CPCB,Delhi Pollution Control Committee,28.646835,77.316032,2016-02-05T14:15:00Z,2026-10-31T18:00:00Z,True
53,6936,"Arya Nagar, Bahadurgarh - HSPCB",CPCB,Haryana State Pollution Control Board,28.670100,76.925400,2019-02-13T00:15:00Z,2026-10-31T18:00:00Z,True
40,5762,"Ashok Vihar, Delhi - DPCC",caaqm,Unknown Governmental Organization,28.695720,77.181295,2018-04-16T15:00:00Z,2018-05-12T06:00:00Z,True
69,8917,"Ashok Vihar, Delhi - DPCC",CPCB,Delhi Pollution Control Committee,28.695381,77.181665,2018-12-15T15:00:00Z,2026-10-31T18:00:00Z,True
17,5570,"Aya Nagar, New Delhi - IMD",CPCB,India Meteorological Department,28.474261,77.131606,2018-03-09T05:30:00Z,2026-10-31T18:00:00Z,True


In [5]:
# Counts by data provider: we keep government monitors only (provider CPCB / caaqm), as the brief asks.
locs["provider.name"].value_counts()

provider.name
CPCB                 57
caaqm                25
N/A                   9
AirGradient           6
StateAir NewDelhi     1
AirNow                1
Name: count, dtype: int64

### 3b · What years does the OpenAQ archive actually hold?
The S3 archive stores one gzipped CSV per station per day under `records/csv.gz/locationid=<id>/year=<y>/month=<m>/`. Before downloading, we list which **months** exist for each of our stations in 2023, 2024 and 2025.

In [6]:
S3 = "https://openaq-data-archive.s3.amazonaws.com"
OPENAQ_IDS = list(STATIONS["openaq_id"]) + EXTRA_OPENAQ_IDS

def months_in_archive(loc_id, year):
    r = SESSION.get(S3, params={"list-type": 2, "prefix": f"records/csv.gz/locationid={loc_id}/year={year}/",
                                "delimiter": "/"}, timeout=60)
    r.raise_for_status()
    return sorted(set(re.findall(r"month=(\d\d)/", r.text)))

cov = []
with ThreadPoolExecutor(8) as ex:
    jobs = {(i, y): ex.submit(months_in_archive, i, y) for i in OPENAQ_IDS for y in DIWALI}
    for (i, y), fut in jobs.items():
        m = fut.result()
        cov.append({"openaq_id": i, "year": y, "months_available": len(m),
                    "has_oct_nov": {"10", "11"} <= set(m)})
coverage = pd.DataFrame(cov).pivot(index="openaq_id", columns="year", values="months_available")
coverage

year,2023,2024,2025
openaq_id,,,
17,0,0,11
50,0,0,11
235,0,0,11
5404,0,0,11
5613,0,0,11
5634,0,0,10
6356,0,0,11
6931,0,0,11
8235,0,0,11


**Finding:** for every Delhi government station, the OpenAQ archive has **no files at all for 2023 or 2024**; data restarts in February 2025. We confirmed the same with the API's `/sensors/{id}/hours` endpoint (0 results for Oct–Nov 2023 and 2024). OpenAQ can therefore give us only the **2025** Diwali. The 2023 and 2024 data come from CPCB directly (Section 5).

In [7]:
# Spot check through the API as well (one station, both missing years)
sensor_id = [s["id"] for s in locs.loc[locs["id"] == 235, "sensors"].iloc[0] if s["parameter"]["name"] == "pm25"]
api_check = []
for sid in sensor_id:
    for y in (2023, 2024):
        r = SESSION.get(f"{OPENAQ_API}/sensors/{sid}/hours",
                        params={"datetime_from": f"{y}-10-01T00:00:00Z", "datetime_to": f"{y}-11-30T23:00:00Z", "limit": 1000},
                        headers={"X-API-Key": OPENAQ_KEY}, timeout=60)
        api_check.append({"location": 235, "sensor_id": sid, "year": y, "hours_returned": len(r.json().get("results", []))})
pd.DataFrame(api_check)

,location,sensor_id,year,hours_returned
0,235,12235610,2023,0
1,235,12235610,2024,0
2,235,384,2023,0
3,235,384,2024,0


### 3c · Download the 2025 daily files
One file per station per day, from 14 Sep to 1 Dec 2025 (one extra day each side because the archive's daily files are cut in UTC while we work in Indian time). Files are saved with source, station and date in the name, e.g. `openaq_loc235_2025-10-20.csv.gz`, and are never modified.

In [8]:
def openaq_day(loc_id, d):
    out = RAW / "openaq" / f"loc{loc_id}" / f"openaq_loc{loc_id}_{d.isoformat()}.csv.gz"
    if out.exists():
        return None
    key = f"records/csv.gz/locationid={loc_id}/year={d.year}/month={d.month:02d}/location-{loc_id}-{d:%Y%m%d}.csv.gz"
    for attempt in range(3):
        try:
            r = SESSION.get(f"{S3}/{key}", timeout=60)
            break
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
    else:
        return {"file": str(out.relative_to(ROOT)), "status": "network error"}
    if r.status_code == 404:
        return {"file": str(out.relative_to(ROOT)), "status": "missing in archive"}
    r.raise_for_status()
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_bytes(r.content)
    return {"file": str(out.relative_to(ROOT)), "status": "ok"}

days_2025 = pd.date_range("2025-09-14", "2025-12-01").date
tasks = [(i, d) for i in OPENAQ_IDS for d in days_2025]
with ThreadPoolExecutor(16) as ex:
    results = [x for x in ex.map(lambda t: openaq_day(*t), tasks) if x]

got = sorted((RAW / "openaq").glob("loc*/*.csv.gz"))
expected_files = {f"openaq_loc{i}_{d.isoformat()}.csv.gz" for i, d in tasks}
not_in_archive = sorted(expected_files - {f.name for f in got})
print(f"Requested {len(tasks)} station-days; files on disk: {len(got)}")
print(f"Station-days that do not exist in the archive: {len(not_in_archive)}")
pd.Series(not_in_archive, name="missing station-day").str.extract(r"loc(?P<openaq_id>\d+)_(?P<date>[\d-]+)").value_counts("openaq_id").rename("days missing").to_frame()

Requested 1027 station-days; files on disk: 1008
Station-days that do not exist in the archive: 19


,days missing
openaq_id,
5634,12
11607,4
50,3


## 4 · Open-Meteo historical weather
One API call per year for a single point in central Delhi (28.61 N, 77.21 E). The weather model behind the archive works on a grid about 9–25 km wide, so every Delhi station falls in essentially the same weather cell; one point represents the city.

Variables: air temperature, relative humidity, rain, wind speed and direction at 10 m, surface pressure and **boundary-layer height** (how deep the layer of air is that pollution mixes into). Times are requested in Indian Standard Time.

In [9]:
METEO_URL = "https://archive-api.open-meteo.com/v1/archive"
METEO_VARS = ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m",
              "wind_direction_10m", "surface_pressure", "boundary_layer_height"]
LAT, LON = 28.61, 77.21

for y, (start, end) in WINDOW.items():
    out = RAW / "openmeteo" / f"openmeteo_delhi_{start}_{end}.json"
    if out.exists():
        continue
    r = SESSION.get(METEO_URL, params={"latitude": LAT, "longitude": LON, "start_date": start, "end_date": end,
                                       "hourly": ",".join(METEO_VARS), "timezone": "Asia/Kolkata"}, timeout=120)
    r.raise_for_status()
    out.write_text(r.text, encoding="utf-8")
    time.sleep(1)

sorted(p.name for p in (RAW / "openmeteo").glob("*.json"))

['openmeteo_delhi_2023-09-15_2023-11-30.json',
 'openmeteo_delhi_2024-09-15_2024-11-30.json',
 'openmeteo_delhi_2025-09-15_2025-11-30.json']

## 5 · CPCB CAAQMS Data Repository (downloaded by hand)
OpenAQ has no 2023–2024 data for Delhi, so we went to the original publisher. The CPCB repository is protected by a captcha, so these files were **downloaded by hand in a web browser**, not by a script. Exact steps:

1. Open <https://airquality.cpcb.gov.in/ccr/#/repository/data> (menu: *Data → CAAQMS Data Repository*).
2. Solve the captcha and press **Verify**.
3. Filters: **State = Delhi**, **City = Delhi**, **Station = (one of the 12 below)**, **Frequency = 1H**, then **Submit**.
4. In the result table, press the download arrow for the rows **2023**, **2024** and **2025**.
5. The site serves the file as `Raw_Data_<year>_site_<id>_<station>_1Hr.csv` (one year of hourly data, all parameters). We renamed each file to `cpcb_site<id>_<station>_<year>.csv` and moved it into `raw/cpcb/`. The content was not changed.

Downloaded on 31 October 2026 (UTC; 31 Oct–1 Nov in Indian time). The cell below checks that all 36 expected files are present.

In [10]:
expected = []
for _, s_ in STATIONS.iterrows():
    for y in DIWALI:
        f = RAW / "cpcb" / f"cpcb_site{s_.cpcb_site_id}_{s_.station_key}_{y}.csv"
        expected.append({"cpcb_name": s_.cpcb_name, "site_id": s_.cpcb_site_id, "year": y,
                         "file": f.name if f.exists() else "MISSING", "kB": round(f.stat().st_size / 1e3) if f.exists() else None})
cpcb_manifest = pd.DataFrame(expected)
n_ok = (cpcb_manifest["file"] != "MISSING").sum()
print(f"CPCB files present: {n_ok} of {len(cpcb_manifest)}")
cpcb_manifest

CPCB files present: 36 of 36


,cpcb_name,site_id,year,file,kB
0,"Anand Vihar, Delhi - DPCC",301,2023,cpcb_site301_anand_vihar_2023.csv,1162
1,"Anand Vihar, Delhi - DPCC",301,2024,cpcb_site301_anand_vihar_2024.csv,997
2,"Anand Vihar, Delhi - DPCC",301,2025,cpcb_site301_anand_vihar_2025.csv,1240
3,"R K Puram, Delhi - DPCC",124,2023,cpcb_site124_rk_puram_2023.csv,1175
4,"R K Puram, Delhi - DPCC",124,2024,cpcb_site124_rk_puram_2024.csv,1099
5,"R K Puram, Delhi - DPCC",124,2025,cpcb_site124_rk_puram_2025.csv,1262
6,"Okhla Phase-2, Delhi - DPCC",1428,2023,cpcb_site1428_okhla_ph2_2023.csv,1299
7,"Okhla Phase-2, Delhi - DPCC",1428,2024,cpcb_site1428_okhla_ph2_2024.csv,1105
8,"Okhla Phase-2, Delhi - DPCC",1428,2025,cpcb_site1428_okhla_ph2_2025.csv,1257
9,"Punjabi Bagh, Delhi - DPCC",125,2023,cpcb_site125_punjabi_bagh_2023.csv,1230


## 6 · First look at every raw file (before any change)
The rubric asks for the shape, data types and the meaning of every kept column for each raw file, before anything is changed. We read one example of each kind and summarise the rest.

### 6a · OpenAQ daily file

In [11]:
ex = RAW / "openaq" / "loc235" / "openaq_loc235_2025-10-20.csv.gz"
oa = pd.read_csv(ex)
print("Example file:", ex.name, "| shape:", oa.shape)
print(oa.dtypes.to_string())
oa.head()

Example file: openaq_loc235_2025-10-20.csv.gz | shape: (785, 9)
location_id      int64
sensors_id       int64
location        object
datetime        object
lat            float64
lon            float64
parameter       object
units           object
value          float64


,location_id,sensors_id,location,datetime,lat,lon,parameter,units,value
0,235,12235609,"Anand Vihar, New Delhi - DPCC-3379575",2025-10-20T01:00:00+05:30,28.646835,77.316032,pm10,µg/m³,1073.0
1,235,12235609,"Anand Vihar, New Delhi - DPCC-3379575",2025-10-20T01:15:00+05:30,28.646835,77.316032,pm10,µg/m³,1073.0
2,235,12235609,"Anand Vihar, New Delhi - DPCC-3379575",2025-10-20T01:30:00+05:30,28.646835,77.316032,pm10,µg/m³,1073.0
3,235,12235609,"Anand Vihar, New Delhi - DPCC-3379575",2025-10-20T01:45:00+05:30,28.646835,77.316032,pm10,µg/m³,1073.0
4,235,12235609,"Anand Vihar, New Delhi - DPCC-3379575",2025-10-20T02:00:00+05:30,28.646835,77.316032,pm10,µg/m³,1014.0


In [12]:
all_oa = pd.concat([pd.read_csv(f) for f in sorted((RAW / "openaq").glob("loc*/*.csv.gz"))], ignore_index=True)
print("All OpenAQ 2025 files combined:", all_oa.shape)
print("Parameters:", all_oa["parameter"].value_counts().to_dict())

openaq_columns = pd.DataFrame([
    ("location_id", "OpenAQ id of the monitoring location", "kept"),
    ("sensors_id", "OpenAQ id of the individual sensor", "kept"),
    ("location", "Location name as OpenAQ writes it (has a numeric suffix)", "kept"),
    ("datetime", "End of the reading's averaging period, ISO 8601 with +05:30 offset", "kept"),
    ("lat", "Latitude (degrees)", "kept"),
    ("lon", "Longitude (degrees)", "kept"),
    ("parameter", "Pollutant or weather variable (pm25, pm10, no2 ...)", "kept; only pm25 used"),
    ("units", "Unit of value (µg/m³ for PM2.5)", "kept"),
    ("value", "Measured value", "kept"),
], columns=["column", "meaning", "use"])
openaq_columns

All OpenAQ 2025 files combined: (803945, 9)
Parameters: {'pm10': 81322, 'no2': 81194, 'pm25': 80525, 'no': 75084, 'o3': 75020, 'co': 74655, 'so2': 67037, 'temperature': 63903, 'relativehumidity': 63896, 'nox': 53609, 'wind_speed': 44056, 'wind_direction': 43644}


,column,meaning,use
0,location_id,OpenAQ id of the monitoring location,kept
1,sensors_id,OpenAQ id of the individual sensor,kept
2,location,Location name as OpenAQ writes it (has a numer...,kept
3,datetime,"End of the reading's averaging period, ISO 860...",kept
4,lat,Latitude (degrees),kept
5,lon,Longitude (degrees),kept
6,parameter,"Pollutant or weather variable (pm25, pm10, no2...",kept; only pm25 used
7,units,Unit of value (µg/m³ for PM2.5),kept
8,value,Measured value,kept


### 6b · Open-Meteo file

In [13]:
mf = sorted((RAW / "openmeteo").glob("*.json"))[0]
mj = json.loads(mf.read_text())
met = pd.DataFrame(mj["hourly"])
print("Example file:", mf.name, "| shape:", met.shape)
print("Units:", mj["hourly_units"])
print("Grid point actually used by the model:", mj["latitude"], mj["longitude"], "| elevation:", mj.get("elevation"))
print(met.dtypes.to_string())
met.head()

Example file: openmeteo_delhi_2023-09-15_2023-11-30.json | shape: (1848, 8)
Units: {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'precipitation': 'mm', 'wind_speed_10m': 'km/h', 'wind_direction_10m': '°', 'surface_pressure': 'hPa', 'boundary_layer_height': 'm'}
Grid point actually used by the model: 28.576448 77.18678 | elevation: 217.0
time                      object
temperature_2m           float64
relative_humidity_2m       int64
precipitation            float64
wind_speed_10m           float64
wind_direction_10m         int64
surface_pressure         float64
boundary_layer_height    float64


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,surface_pressure,boundary_layer_height
0,2023-09-15T00:00,26.5,90,0.0,4.4,189,980.5,180.0
1,2023-09-15T01:00,26.2,92,0.0,2.2,279,979.8,210.0
2,2023-09-15T02:00,26.0,94,0.0,4.0,95,979.1,260.0
3,2023-09-15T03:00,25.6,96,0.0,3.0,104,979.1,195.0
4,2023-09-15T04:00,25.5,97,0.0,2.5,352,979.3,20.0


In [14]:
meteo_columns = pd.DataFrame([
    ("time", "Hour (local time, Asia/Kolkata), start of hour", "kept"),
    ("temperature_2m", "Air temperature 2 m above ground (°C)", "kept"),
    ("relative_humidity_2m", "Relative humidity at 2 m (%)", "kept"),
    ("precipitation", "Rain in the preceding hour (mm)", "kept"),
    ("wind_speed_10m", "Wind speed 10 m above ground (km/h)", "kept"),
    ("wind_direction_10m", "Direction the wind blows FROM (degrees, 0 = north)", "kept"),
    ("surface_pressure", "Air pressure at the surface (hPa)", "kept"),
    ("boundary_layer_height", "Depth of the mixed layer of air near the ground (m)", "kept"),
], columns=["column", "meaning", "use"])
meteo_columns

,column,meaning,use
0,time,"Hour (local time, Asia/Kolkata), start of hour",kept
1,temperature_2m,Air temperature 2 m above ground (°C),kept
2,relative_humidity_2m,Relative humidity at 2 m (%),kept
3,precipitation,Rain in the preceding hour (mm),kept
4,wind_speed_10m,Wind speed 10 m above ground (km/h),kept
5,wind_direction_10m,"Direction the wind blows FROM (degrees, 0 = no...",kept
6,surface_pressure,Air pressure at the surface (hPa),kept
7,boundary_layer_height,Depth of the mixed layer of air near the groun...,kept


### 6c · CPCB yearly file

In [15]:
cp_files = sorted((RAW / "cpcb").glob("cpcb_site*.csv"))
if cp_files:
    cp = pd.read_csv(cp_files[0])
    print("Example file:", cp_files[0].name, "| shape:", cp.shape)
    print(cp.dtypes.to_string())
    display(cp.head())
    shapes = pd.DataFrame([{"file": f.name, "rows": len(pd.read_csv(f, usecols=[0])),
                            "columns": len(pd.read_csv(f, nrows=0).columns)} for f in cp_files])
    display(shapes)
else:
    print("No CPCB files yet - see Section 5")

Example file: cpcb_site107_pusa_iitm_2023.csv | shape: (8760, 25)
Timestamp               object
PM2.5 (µg/m³)          float64
PM10 (µg/m³)           float64
NO (µg/m³)             float64
NO2 (µg/m³)            float64
NOx (ppb)              float64
NH3 (µg/m³)            float64
SO2 (µg/m³)            float64
CO (mg/m³)             float64
Ozone (µg/m³)          float64
Benzene (µg/m³)        float64
Toluene (µg/m³)        float64
Xylene (µg/m³)         float64
O Xylene (µg/m³)       float64
Eth-Benzene (µg/m³)    float64
MP-Xylene (µg/m³)      float64
AT (°C)                float64
RH (%)                 float64
WS (m/s)               float64
WD (deg)               float64
RF (mm)                float64
TOT-RF (mm)            float64
SR (W/mt2)             float64
BP (mmHg)              float64
VWS (m/s)              float64


,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),Xylene (µg/m³),O Xylene (µg/m³),Eth-Benzene (µg/m³),MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2023-01-01 00:00:00,134.3750,198.9650,23.4000,34.0000,57.3950,NaN,NaN,1.0025,25.1150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2023-01-01 01:00:00,131.6075,194.1425,19.9700,29.7300,49.7025,NaN,NaN,0.9650,24.8850,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2023-01-01 02:00:00,147.6050,187.4325,19.8200,27.4975,47.3100,NaN,NaN,0.9350,25.4575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2023-01-01 03:00:00,122.8325,198.4325,22.6300,26.1950,48.8075,NaN,NaN,0.9125,25.0150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2023-01-01 04:00:00,96.7700,168.8925,36.4075,30.5625,66.9675,NaN,NaN,0.8850,24.6450,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


,file,rows,columns
0,cpcb_site107_pusa_iitm_2023.csv,8760,25
1,cpcb_site107_pusa_iitm_2024.csv,8784,25
2,cpcb_site107_pusa_iitm_2025.csv,8760,25
3,cpcb_site109_lodhi_road_imd_2023.csv,8760,25
4,cpcb_site109_lodhi_road_imd_2024.csv,8784,25
5,cpcb_site109_lodhi_road_imd_2025.csv,8760,25
6,cpcb_site117_ito_2023.csv,8760,25
7,cpcb_site117_ito_2024.csv,8784,25
8,cpcb_site117_ito_2025.csv,8760,25
9,cpcb_site124_rk_puram_2023.csv,8760,25


In [16]:
cpcb_columns = pd.DataFrame([
    ("Timestamp", "Start of the hour, local time (IST), no time-zone marker", "kept"),
    ("PM2.5 (µg/m³)", "Hourly mean PM2.5 concentration", "kept - main variable"),
    ("PM10 (µg/m³)", "Hourly mean PM10", "kept for context"),
    ("NO, NO2, NOx, NH3, SO2, CO, Ozone", "Other gases", "dropped - outside the question"),
    ("Benzene, Toluene, Xylene, ...", "Volatile organic compounds", "dropped - outside the question"),
    ("AT (°C)", "Air temperature measured at the station", "kept - cross-check with Open-Meteo"),
    ("RH (%)", "Relative humidity at the station", "kept - cross-check"),
    ("WS (m/s)", "Wind speed at the station", "kept - cross-check"),
    ("WD (deg)", "Wind direction at the station", "kept - cross-check"),
    ("RF, TOT-RF (mm)", "Rainfall", "dropped - Open-Meteo rain used"),
    ("SR (W/mt2)", "Solar radiation", "dropped"),
    ("BP (mmHg)", "Barometric pressure", "dropped"),
    ("VWS (m/s)", "Vertical wind speed", "dropped"),
], columns=["column", "meaning", "use"])
cpcb_columns

,column,meaning,use
0,Timestamp,"Start of the hour, local time (IST), no time-z...",kept
1,PM2.5 (µg/m³),Hourly mean PM2.5 concentration,kept - main variable
2,PM10 (µg/m³),Hourly mean PM10,kept for context
3,"NO, NO2, NOx, NH3, SO2, CO, Ozone",Other gases,dropped - outside the question
4,"Benzene, Toluene, Xylene, ...",Volatile organic compounds,dropped - outside the question
5,AT (°C),Air temperature measured at the station,kept - cross-check with Open-Meteo
6,RH (%),Relative humidity at the station,kept - cross-check
7,WS (m/s),Wind speed at the station,kept - cross-check
8,WD (deg),Wind direction at the station,kept - cross-check
9,"RF, TOT-RF (mm)",Rainfall,dropped - Open-Meteo rain used


## 7 · Summary of what was collected

In [17]:
def folder_size(pattern):
    files = list(RAW.glob(pattern))
    return len(files), round(sum(f.stat().st_size for f in files) / 1e6, 1)
summary = pd.DataFrame([
    ("CPCB CAAQMS Data Repository", "https://airquality.cpcb.gov.in/ccr/#/repository/data", "by hand (captcha)", "Government of India public data; no licence stated", *folder_size("cpcb/*.csv")),
    ("OpenAQ API v3 (station list)", "https://api.openaq.org/v3/locations", "script", "OpenAQ terms of use", *folder_size("openaq/*.json")),
    ("OpenAQ AWS archive (2025 readings)", "https://openaq-data-archive.s3.amazonaws.com", "script", "OpenAQ terms of use", *folder_size("openaq/loc*/*.csv.gz")),
    ("Open-Meteo Historical Weather API", "https://archive-api.open-meteo.com/v1/archive", "script", "CC BY 4.0", *folder_size("openmeteo/*.json")),
], columns=["source", "url", "how", "licence", "files", "MB"])
summary.insert(3, "accessed", "31 Oct 2026 (UTC)")
summary

,source,url,how,accessed,licence,files,MB
0,CPCB CAAQMS Data Repository,https://airquality.cpcb.gov.in/ccr/#/repositor...,by hand (captcha),31 Oct 2026 (UTC),Government of India public data; no licence st...,36,40.4
1,OpenAQ API v3 (station list),https://api.openaq.org/v3/locations,script,31 Oct 2026 (UTC),OpenAQ terms of use,1,0.2
2,OpenAQ AWS archive (2025 readings),https://openaq-data-archive.s3.amazonaws.com,script,31 Oct 2026 (UTC),OpenAQ terms of use,1008,4.7
3,Open-Meteo Historical Weather API,https://archive-api.open-meteo.com/v1/archive,script,31 Oct 2026 (UTC),CC BY 4.0,3,0.3
